In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('Metro_Interstate_Traffic_Volume.csv', sep=';')
print("Shape:", df.shape)
print("\nPremières lignes :")
df.head()

In [ ]:
import os
print(os.listdir())

In [ ]:
import os
os.chdir('Projet_Trafic_Casa')
print(os.listdir())

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('Metro_Interstate_Traffic_Volume.csv', sep=';')
print("Shape:", df.shape)
print("\nPremières lignes :")
df.head()

In [ ]:
print("Types des colonnes :")
print(df.dtypes)
print("\nValeurs manquantes :")
print(df.isnull().sum())
print("\nStatistiques générales :")
df.describe()

In [ ]:
# Corriger la colonne holiday (remplacer NaN par "None")
df['holiday'] = df['holiday'].fillna('None')

# Conversion de la date
df['date_time'] = pd.to_datetime(df['date_time'], dayfirst=True)

# Extraction des features temporelles
df['hour']        = df['date_time'].dt.hour
df['day_of_week'] = df['date_time'].dt.dayofweek
df['month']       = df['date_time'].dt.month
df['year']        = df['date_time'].dt.year

# Nouvelles colonnes utiles
df['is_holiday']   = (df['holiday'] != 'None').astype(int)
df['temp_celsius'] = df['temp'] - 273.15
df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)

# Supprimer les températures impossibles (temp = 0 Kelvin)
avant = len(df)
df = df[df['temp'] > 200]
print(f"Lignes supprimées (temp incorrecte) : {avant - len(df)}")

# Suppression des doublons
avant2 = len(df)
df.drop_duplicates(inplace=True)
print(f"Doublons supprimés : {avant2 - len(df)}")

# Vérification finale
print(f"\nDataset final : {df.shape[0]} lignes, {df.shape[1]} colonnes")
df[['date_time','hour','day_of_week','month','is_holiday','temp_celsius','is_weekend','traffic_volume']].head(10)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 4))
df.groupby('hour')['traffic_volume'].mean().plot(kind='bar', color='steelblue')
plt.title('Trafic moyen par heure de la journée')
plt.xlabel('Heure')
plt.ylabel('Volume de trafic moyen')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
jours = ['Lundi','Mardi','Mercredi','Jeudi','Vendredi','Samedi','Dimanche']

plt.figure(figsize=(10, 4))
df.groupby('day_of_week')['traffic_volume'].mean().plot(kind='bar', color='coral')
plt.title('Trafic moyen par jour de la semaine')
plt.xlabel('Jour')
plt.ylabel('Volume de trafic moyen')
plt.xticks(ticks=range(7), labels=jours, rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))
df.groupby('weather_main')['traffic_volume'].mean().sort_values().plot(kind='barh', color='mediumseagreen')
plt.title('Trafic moyen par condition météo')
plt.xlabel('Volume de trafic moyen')
plt.tight_layout()
plt.show()

In [ ]:
pivot = df.pivot_table(values='traffic_volume', index='day_of_week', columns='hour', aggfunc='mean')
pivot.index = ['Lundi','Mardi','Mercredi','Jeudi','Vendredi','Samedi','Dimanche']

plt.figure(figsize=(16, 5))
sns.heatmap(pivot, cmap='YlOrRd', fmt='.0f', annot=False)
plt.title('Carte thermique — Trafic moyen par heure et jour')
plt.xlabel('Heure')
plt.ylabel('Jour')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encoder la colonne weather_main (texte → chiffre)
le = LabelEncoder()
df['weather_encoded'] = le.fit_transform(df['weather_main'])

# Définir les features (X) et le label (y)
features = ['hour', 'day_of_week', 'month', 'temp_celsius',
            'rain_1h', 'snow_1h', 'clouds_all',
            'is_holiday', 'is_weekend', 'weather_encoded']

X = df[features]
y = df['traffic_volume']

print("Features shape:", X.shape)
print("Label shape:", y.shape)
print("\nAperçu des features :")
X.head()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Entraînement : {X_train.shape[0]} lignes")
print(f"Test         : {X_test.shape[0]} lignes")

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
preds_rf = rf.predict(X_test)

mae_rf  = mean_absolute_error(y_test, preds_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, preds_rf))
r2_rf   = r2_score(y_test, preds_rf)

print("=== Random Forest ===")
print(f"MAE  : {mae_rf:.2f}")
print(f"RMSE : {rmse_rf:.2f}")
print(f"R²   : {r2_rf:.4f}")

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
xgb.fit(X_train, y_train)
preds_xgb = xgb.predict(X_test)

mae_xgb  = mean_absolute_error(y_test, preds_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, preds_xgb))
r2_xgb   = r2_score(y_test, preds_xgb)

print("=== XGBoost ===")
print(f"MAE  : {mae_xgb:.2f}")
print(f"RMSE : {rmse_xgb:.2f}")
print(f"R²   : {r2_xgb:.4f}")

In [ ]:
resultats = pd.DataFrame({
    'Modèle': ['Random Forest', 'XGBoost'],
    'MAE':    [round(mae_rf, 2),  round(mae_xgb, 2)],
    'RMSE':   [round(rmse_rf, 2), round(rmse_xgb, 2)],
    'R²':     [round(r2_rf, 4),   round(r2_xgb, 4)]
})
print(resultats)

In [ ]:
!pip install xgboost

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
xgb.fit(X_train, y_train)
preds_xgb = xgb.predict(X_test)

mae_xgb  = mean_absolute_error(y_test, preds_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, preds_xgb))
r2_xgb   = r2_score(y_test, preds_xgb)

print("=== XGBoost ===")
print(f"MAE  : {mae_xgb:.2f}")
print(f"RMSE : {rmse_xgb:.2f}")
print(f"R²   : {r2_xgb:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Récupérer l'importance des features depuis XGBoost
importances = xgb.feature_importances_
indices = np.argsort(importances)[::-1]
noms = [features[i] for i in indices]

plt.figure(figsize=(10, 5))
plt.bar(noms, importances[indices], color='steelblue')
plt.title('Importance des features — XGBoost')
plt.xlabel('Features')
plt.ylabel('Importance')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print("\nClassement :")
for i, idx in enumerate(indices):
    print(f"{i+1}. {features[idx]} → {importances[idx]:.4f}")

In [ ]:
plt.figure(figsize=(10, 5))

# Prendre 200 exemples pour la lisibilité
y_sample = y_test[:200].values
p_sample = preds_xgb[:200]

plt.plot(y_sample, label='Réel', color='steelblue', alpha=0.7)
plt.plot(p_sample, label='Prédit', color='coral', alpha=0.7, linestyle='--')
plt.title('Trafic réel vs prédit — XGBoost (200 premiers exemples)')
plt.xlabel('Échantillons')
plt.ylabel('Volume de trafic')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, preds_xgb, alpha=0.3, color='steelblue', s=5)
plt.plot([0, 7000], [0, 7000], color='red', linestyle='--', label='Prédiction parfaite')
plt.title('Réel vs Prédit — XGBoost')
plt.xlabel('Valeurs réelles')
plt.ylabel('Valeurs prédites')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import joblib

joblib.dump(xgb, 'modele_trafic_xgboost.pkl')
print("Modèle sauvegardé ✅")

In [ ]:
!pip install dash plotly 

In [ ]:
import subprocess
subprocess.Popen(['python', 'dashboard.py'])

import time
time.sleep(3)
print("Dashboard lancé ! Ouvre ton navigateur sur : http://127.0.0.1:8050")

In [ ]:
import subprocess
import time

process = subprocess.Popen(
    ['python', 'dashboard.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(4)
out, err = process.communicate(timeout=5)
print("OUTPUT:", out.decode())
print("ERREUR:", err.decode())

In [1]:
# Créer le fichier dashboard.py
code = '''
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("Metro_Interstate_Traffic_Volume.csv", sep=";")
df["holiday"] = df["holiday"].fillna("None")
df["date_time"] = pd.to_datetime(df["date_time"], dayfirst=True)
df["hour"] = df["date_time"].dt.hour
df["day_of_week"] = df["date_time"].dt.dayofweek
df["month"] = df["date_time"].dt.month
df["is_holiday"] = (df["holiday"] != "None").astype(int)
df["temp_celsius"] = df["temp"] - 273.15
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
df = df[df["temp"] > 200]
df.drop_duplicates(inplace=True)
le = LabelEncoder()
df["weather_encoded"] = le.fit_transform(df["weather_main"])
model = joblib.load("modele_trafic_xgboost.pkl")
jours = ["Lundi","Mardi","Mercredi","Jeudi","Vendredi","Samedi","Dimanche"]
meteos = sorted(df["weather_main"].unique())
app = dash.Dash(__name__)
app.layout = html.Div(style={"fontFamily":"Arial","padding":"20px","backgroundColor":"#f8f9fa"}, children=[
    html.H1("Dashboard Trafic Routier", style={"textAlign":"center","color":"#2c3e50"}),
    html.Hr(),
    html.H2("Analyse du trafic"),
    html.Div([
        html.Label("Filtrer par météo :"),
        dcc.Dropdown(id="meteo-filter",
            options=[{"label":m,"value":m} for m in meteos],
            value=None, placeholder="Toutes les conditions", clearable=True)
    ], style={"width":"40%","marginBottom":"20px"}),
    html.Div([
        dcc.Graph(id="graph-heure", style={"width":"50%","display":"inline-block"}),
        dcc.Graph(id="graph-jour", style={"width":"50%","display":"inline-block"}),
    ]),
    dcc.Graph(id="graph-heatmap"),
    html.Hr(),
    html.H2("Prediction en temps reel"),
    html.Div([
        html.Label("Heure :"),
        dcc.Slider(0,23,1,value=8,marks={i:str(i) for i in range(0,24,2)},id="input-hour"),
        html.Br(),
        html.Label("Jour :"),
        dcc.Dropdown(id="input-day",
            options=[{"label":jours[i],"value":i} for i in range(7)],
            value=0, style={"width":"30%"}),
        html.Br(),
        html.Label("Mois :"),
        dcc.Slider(1,12,1,value=6,marks={i:str(i) for i in range(1,13)},id="input-month"),
        html.Br(),
        html.Label("Temperature (C) :"),
        dcc.Slider(-20,40,1,value=15,marks={i:str(i) for i in range(-20,41,10)},id="input-temp"),
        html.Br(),
        html.Label("Meteo :"),
        dcc.Dropdown(id="input-meteo",
            options=[{"label":m,"value":m} for m in meteos],
            value="Clear", style={"width":"30%"}),
        html.Br(),
        html.Label("Jour ferie ?"),
        dcc.RadioItems(id="input-holiday",
            options=[{"label":"Non","value":0},{"label":"Oui","value":1}],
            value=0, inline=True),
        html.Br(),
        html.Div(id="prediction-output",
            style={"fontSize":"24px","fontWeight":"bold","color":"#27ae60",
                   "textAlign":"center","padding":"20px","backgroundColor":"white",
                   "borderRadius":"10px","border":"2px solid #27ae60"})
    ], style={"backgroundColor":"white","padding":"20px","borderRadius":"10px"}),
])

@app.callback(
    Output("graph-heure","figure"),
    Output("graph-jour","figure"),
    Output("graph-heatmap","figure"),
    Input("meteo-filter","value"))
def update_graphs(meteo):
    dff = df[df["weather_main"]==meteo] if meteo else df
    fig1 = px.bar(dff.groupby("hour")["traffic_volume"].mean().reset_index(),
        x="hour",y="traffic_volume",title="Trafic moyen par heure",
        color_discrete_sequence=["#3498db"])
    fig2 = px.bar(dff.groupby("day_of_week")["traffic_volume"].mean().reset_index(),
        x="day_of_week",y="traffic_volume",title="Trafic moyen par jour",
        color_discrete_sequence=["#e74c3c"])
    fig2.update_xaxes(tickvals=list(range(7)),ticktext=jours)
    pivot = dff.pivot_table(values="traffic_volume",index="day_of_week",columns="hour",aggfunc="mean")
    fig3 = px.imshow(pivot,title="Carte thermique",color_continuous_scale="YlOrRd",y=jours)
    return fig1,fig2,fig3

@app.callback(
    Output("prediction-output","children"),
    Input("input-hour","value"),
    Input("input-day","value"),
    Input("input-month","value"),
    Input("input-temp","value"),
    Input("input-meteo","value"),
    Input("input-holiday","value"))
def predict(hour,day,month,temp,meteo,holiday):
    is_weekend = 1 if day >= 5 else 0
    weather_enc = le.transform([meteo])[0] if meteo in le.classes_ else 0
    X_input = pd.DataFrame([[hour,day,month,temp,0,0,50,holiday,is_weekend,weather_enc]],
        columns=["hour","day_of_week","month","temp_celsius",
                 "rain_1h","snow_1h","clouds_all","is_holiday","is_weekend","weather_encoded"])
    pred = model.predict(X_input)[0]
    return f"Trafic predit : {int(pred):,} vehicules/heure"

if __name__ == "__main__":
    app.run(debug=False, port=8050)
'''

with open('dashboard.py', 'w', encoding='utf-8') as f:
    f.write(code)
print("✅ Fichier créé !")

✅ Fichier créé !


In [2]:
import subprocess
import time

process = subprocess.Popen(
    ['python', 'dashboard.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(8)

out, err = process.communicate(timeout=15)
print("OUTPUT:", out.decode('utf-8'))
print("ERREUR:", err.decode('utf-8'))

OUTPUT: 
ERREUR: Traceback (most recent call last):
  File "C:\Users\user\dashboard.py", line 2, in <module>
    import dash
  File "C:\Users\user\OneDrive\Attachments\Lib\site-packages\dash\__init__.py", line 38, in <module>
    from .dash import (  # noqa: F401,E402
  File "C:\Users\user\OneDrive\Attachments\Lib\site-packages\dash\dash.py", line 83, in <module>
    from ._jupyter import jupyter_dash, JupyterDisplayMode
  File "C:\Users\user\OneDrive\Attachments\Lib\site-packages\dash\_jupyter.py", line 28, in <module>
    _dash_comm = create_comm(target_name="dash")  # type: ignore[misc]
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\OneDrive\Attachments\Lib\site-packages\comm\__init__.py", line 27, in _create_comm
    raise NotImplementedError("Cannot ")
NotImplementedError: Cannot 



In [3]:
import sys
import subprocess

# Installer dash avec le bon Python (celui d'Anaconda)
subprocess.run([sys.executable, '-m', 'pip', 'install', 
                'dash', 'plotly', '--upgrade'], 
               capture_output=False)
print("✅ Installation terminée !")

✅ Installation terminée !


In [4]:
import subprocess
import sys
import time

process = subprocess.Popen(
    [sys.executable, 'dashboard.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(8)
out, err = process.communicate(timeout=15)
print("OUTPUT:", out.decode('utf-8'))
print("ERREUR:", err.decode('utf-8'))

OUTPUT: 
ERREUR: Traceback (most recent call last):
  File "C:\Users\user\dashboard.py", line 2, in <module>
    import dash
  File "C:\Users\user\OneDrive\Attachments\Lib\site-packages\dash\__init__.py", line 38, in <module>
    from .dash import (  # noqa: F401,E402
  File "C:\Users\user\OneDrive\Attachments\Lib\site-packages\dash\dash.py", line 83, in <module>
    from ._jupyter import jupyter_dash, JupyterDisplayMode
  File "C:\Users\user\OneDrive\Attachments\Lib\site-packages\dash\_jupyter.py", line 28, in <module>
    _dash_comm = create_comm(target_name="dash")  # type: ignore[misc]
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\OneDrive\Attachments\Lib\site-packages\comm\__init__.py", line 27, in _create_comm
    raise NotImplementedError("Cannot ")
NotImplementedError: Cannot 



In [5]:
import sys
import os

# Vérifier quel Python on utilise
print("Python utilisé:", sys.executable)

# Créer le dashboard dans le BON dossier
os.chdir('C:\\Users\\user\\Projet_Trafic_Casa')
print("Dossier actuel:", os.getcwd())

# Désinstaller le mauvais dash et réinstaller
os.system(f'"{sys.executable}" -m pip uninstall dash -y')
os.system(f'"{sys.executable}" -m pip install dash plotly --upgrade')
print("✅ Dash réinstallé !")

Python utilisé: C:\Users\user\OneDrive\Attachments\python.exe
Dossier actuel: C:\Users\user\Projet_Trafic_Casa
✅ Dash réinstallé !


In [6]:
import os

# Chemin du Python Anaconda
python_anaconda = 'C:\\Users\\user\\anaconda3\\python.exe'

# Vérifier qu'il existe
if os.path.exists(python_anaconda):
    print("✅ Python Anaconda trouvé !")
else:
    print("❌ Pas trouvé, on cherche...")
    # Chercher Anaconda
    for path in [
        'C:\\Users\\user\\anaconda3\\python.exe',
        'C:\\ProgramData\\anaconda3\\python.exe',
        'C:\\Users\\user\\Anaconda3\\python.exe',
        'C:\\ProgramData\\Anaconda3\\python.exe',
    ]:
        if os.path.exists(path):
            print("✅ Trouvé ici:", path)
            break

✅ Python Anaconda trouvé !


In [7]:
import subprocess
import time
import os

python_anaconda = 'C:\\Users\\user\\anaconda3\\python.exe'

# Installer dash avec le bon Python
print("Installation de dash...")
os.system(f'"{python_anaconda}" -m pip install dash plotly -q')
print("✅ Dash installé !")

# Lancer le dashboard
process = subprocess.Popen(
    [python_anaconda, 'dashboard.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(8)
out, err = process.communicate(timeout=20)
print("OUTPUT:", out.decode('utf-8'))
print("ERREUR:", err.decode('utf-8'))

Installation de dash...
✅ Dash installé !
OUTPUT: 
ERREUR: Traceback (most recent call last):
  File "C:\Users\user\Projet_Trafic_Casa\dashboard.py", line 1, in <module>
    test
NameError: name 'test' is not defined



In [8]:
code = '''import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("Metro_Interstate_Traffic_Volume.csv", sep=";")
df["holiday"] = df["holiday"].fillna("None")
df["date_time"] = pd.to_datetime(df["date_time"], dayfirst=True)
df["hour"] = df["date_time"].dt.hour
df["day_of_week"] = df["date_time"].dt.dayofweek
df["month"] = df["date_time"].dt.month
df["is_holiday"] = (df["holiday"] != "None").astype(int)
df["temp_celsius"] = df["temp"] - 273.15
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
df = df[df["temp"] > 200]
df.drop_duplicates(inplace=True)
le = LabelEncoder()
df["weather_encoded"] = le.fit_transform(df["weather_main"])
model = joblib.load("modele_trafic_xgboost.pkl")
jours = ["Lundi","Mardi","Mercredi","Jeudi","Vendredi","Samedi","Dimanche"]
meteos = sorted(df["weather_main"].unique())
app = dash.Dash(__name__)
app.layout = html.Div(style={"fontFamily":"Arial","padding":"20px","backgroundColor":"#f8f9fa"}, children=[
    html.H1("Dashboard Trafic Routier", style={"textAlign":"center","color":"#2c3e50"}),
    html.Hr(),
    html.H2("Analyse du trafic"),
    html.Div([
        html.Label("Filtrer par meteo :"),
        dcc.Dropdown(id="meteo-filter",
            options=[{"label":m,"value":m} for m in meteos],
            value=None, placeholder="Toutes les conditions", clearable=True)
    ], style={"width":"40%","marginBottom":"20px"}),
    html.Div([
        dcc.Graph(id="graph-heure", style={"width":"50%","display":"inline-block"}),
        dcc.Graph(id="graph-jour", style={"width":"50%","display":"inline-block"}),
    ]),
    dcc.Graph(id="graph-heatmap"),
    html.Hr(),
    html.H2("Prediction en temps reel"),
    html.Div([
        html.Label("Heure :"),
        dcc.Slider(0,23,1,value=8,marks={i:str(i) for i in range(0,24,2)},id="input-hour"),
        html.Br(),
        html.Label("Jour :"),
        dcc.Dropdown(id="input-day",
            options=[{"label":jours[i],"value":i} for i in range(7)],
            value=0, style={"width":"30%"}),
        html.Br(),
        html.Label("Mois :"),
        dcc.Slider(1,12,1,value=6,marks={i:str(i) for i in range(1,13)},id="input-month"),
        html.Br(),
        html.Label("Temperature (C) :"),
        dcc.Slider(-20,40,1,value=15,marks={i:str(i) for i in range(-20,41,10)},id="input-temp"),
        html.Br(),
        html.Label("Meteo :"),
        dcc.Dropdown(id="input-meteo",
            options=[{"label":m,"value":m} for m in meteos],
            value="Clear", style={"width":"30%"}),
        html.Br(),
        html.Label("Jour ferie ?"),
        dcc.RadioItems(id="input-holiday",
            options=[{"label":"Non","value":0},{"label":"Oui","value":1}],
            value=0, inline=True),
        html.Br(),
        html.Div(id="prediction-output",
            style={"fontSize":"24px","fontWeight":"bold","color":"#27ae60",
                   "textAlign":"center","padding":"20px","backgroundColor":"white",
                   "borderRadius":"10px","border":"2px solid #27ae60"})
    ], style={"backgroundColor":"white","padding":"20px","borderRadius":"10px"}),
])

@app.callback(
    Output("graph-heure","figure"),
    Output("graph-jour","figure"),
    Output("graph-heatmap","figure"),
    Input("meteo-filter","value"))
def update_graphs(meteo):
    dff = df[df["weather_main"]==meteo] if meteo else df
    fig1 = px.bar(dff.groupby("hour")["traffic_volume"].mean().reset_index(),
        x="hour",y="traffic_volume",title="Trafic moyen par heure",
        color_discrete_sequence=["#3498db"])
    fig2 = px.bar(dff.groupby("day_of_week")["traffic_volume"].mean().reset_index(),
        x="day_of_week",y="traffic_volume",title="Trafic moyen par jour",
        color_discrete_sequence=["#e74c3c"])
    fig2.update_xaxes(tickvals=list(range(7)),ticktext=jours)
    pivot = dff.pivot_table(values="traffic_volume",index="day_of_week",columns="hour",aggfunc="mean")
    fig3 = px.imshow(pivot,title="Carte thermique",color_continuous_scale="YlOrRd",y=jours)
    return fig1,fig2,fig3

@app.callback(
    Output("prediction-output","children"),
    Input("input-hour","value"),
    Input("input-day","value"),
    Input("input-month","value"),
    Input("input-temp","value"),
    Input("input-meteo","value"),
    Input("input-holiday","value"))
def predict(hour,day,month,temp,meteo,holiday):
    is_weekend = 1 if day >= 5 else 0
    weather_enc = le.transform([meteo])[0] if meteo in le.classes_ else 0
    X_input = pd.DataFrame([[hour,day,month,temp,0,0,50,holiday,is_weekend,weather_enc]],
        columns=["hour","day_of_week","month","temp_celsius",
                 "rain_1h","snow_1h","clouds_all","is_holiday","is_weekend","weather_encoded"])
    pred = model.predict(X_input)[0]
    return f"Trafic predit : {int(pred):,} vehicules/heure"

if __name__ == "__main__":
    app.run(debug=False, port=8050)
'''

with open('dashboard.py', 'w', encoding='utf-8') as f:
    f.write(code)
print("✅ Fichier dashboard.py créé correctement !")

✅ Fichier dashboard.py créé correctement !


In [9]:
import subprocess
import time

python_anaconda = 'C:\\Users\\user\\anaconda3\\python.exe'

process = subprocess.Popen(
    [python_anaconda, 'dashboard.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(10)
out, err = process.communicate(timeout=20)
print("OUTPUT:", out.decode('utf-8'))
print("ERREUR:", err.decode('utf-8'))

OUTPUT: 
ERREUR: Traceback (most recent call last):
  File "C:\Users\user\Projet_Trafic_Casa\dashboard.py", line 1, in <module>
    import dash
  File "C:\Users\user\anaconda3\Lib\site-packages\dash\__init__.py", line 38, in <module>
    from .dash import (  # noqa: F401,E402
  File "C:\Users\user\anaconda3\Lib\site-packages\dash\dash.py", line 83, in <module>
    from ._jupyter import jupyter_dash, JupyterDisplayMode
  File "C:\Users\user\anaconda3\Lib\site-packages\dash\_jupyter.py", line 28, in <module>
    _dash_comm = create_comm(target_name="dash")  # type: ignore[misc]
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\anaconda3\Lib\site-packages\comm\__init__.py", line 27, in _create_comm
    raise NotImplementedError("Cannot ")
NotImplementedError: Cannot 



In [10]:
import os

python_anaconda = 'C:\\Users\\user\\anaconda3\\python.exe'

# Désinstaller et réinstaller les bonnes versions
os.system(f'"{python_anaconda}" -m pip uninstall dash comm -y')
os.system(f'"{python_anaconda}" -m pip install dash==2.14.0 comm==0.1.4')
print("✅ Versions corrigées !")

✅ Versions corrigées !


In [11]:
import subprocess
import time

python_anaconda = 'C:\\Users\\user\\anaconda3\\python.exe'

process = subprocess.Popen(
    [python_anaconda, 'dashboard.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(10)
out, err = process.communicate(timeout=20)
print("OUTPUT:", out.decode('utf-8'))
print("ERREUR:", err.decode('utf-8'))

OUTPUT: 
ERREUR: Traceback (most recent call last):
  File "C:\Users\user\Projet_Trafic_Casa\dashboard.py", line 21, in <module>
    model = joblib.load("modele_trafic_xgboost.pkl")
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\anaconda3\Lib\site-packages\joblib\numpy_pickle.py", line 658, in load
    obj = _unpickle(fobj, filename, mmap_mode)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\anaconda3\Lib\site-packages\joblib\numpy_pickle.py", line 577, in _unpickle
    obj = unpickler.load()
          ^^^^^^^^^^^^^^^^
  File "C:\Users\user\anaconda3\Lib\pickle.py", line 1213, in load
    dispatch[key[0]](self)
  File "C:\Users\user\anaconda3\Lib\pickle.py", line 1538, in load_stack_global
    self.append(self.find_class(module, name))
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\anaconda3\Lib\pickle.py", line 1580, in find_class
    __import__(module, level=0)
ModuleNotFoundError: No module named 'xgboost'



In [12]:
import os

python_anaconda = 'C:\\Users\\user\\anaconda3\\python.exe'

os.system(f'"{python_anaconda}" -m pip install xgboost joblib scikit-learn')
print("✅ XGBoost installé !")

✅ XGBoost installé !


In [13]:
import subprocess
import time

python_anaconda = 'C:\\Users\\user\\anaconda3\\python.exe'

process = subprocess.Popen(
    [python_anaconda, 'dashboard.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(10)
out, err = process.communicate(timeout=20)
print("OUTPUT:", out.decode('utf-8'))
print("ERREUR:", err.decode('utf-8'))

TimeoutExpired: Command '['C:\\Users\\user\\anaconda3\\python.exe', 'dashboard.py']' timed out after 20 seconds

In [14]:
code = '''import dash
from dash import dcc, html, Input, Output, State
import plotly.express as px
import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("Metro_Interstate_Traffic_Volume.csv", sep=";")
df["holiday"] = df["holiday"].fillna("None")
df["date_time"] = pd.to_datetime(df["date_time"], dayfirst=True)
df["hour"] = df["date_time"].dt.hour
df["day_of_week"] = df["date_time"].dt.dayofweek
df["month"] = df["date_time"].dt.month
df["is_holiday"] = (df["holiday"] != "None").astype(int)
df["temp_celsius"] = df["temp"] - 273.15
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
df = df[df["temp"] > 200]
df.drop_duplicates(inplace=True)
le = LabelEncoder()
df["weather_encoded"] = le.fit_transform(df["weather_main"])
model = joblib.load("modele_trafic_xgboost.pkl")
jours = ["Lundi","Mardi","Mercredi","Jeudi","Vendredi","Samedi","Dimanche"]
jours_en = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
meteos = sorted(df["weather_main"].unique())

def analyser_question(question):
    q = question.lower().strip()
    
    # Stats générales
    trafic_moyen = int(df["traffic_volume"].mean())
    trafic_max = int(df["traffic_volume"].max())
    trafic_min = int(df["traffic_volume"].min())
    
    # Meilleure heure
    meilleure_heure = df.groupby("hour")["traffic_volume"].mean().idxmin()
    pire_heure = df.groupby("hour")["traffic_volume"].mean().idxmax()
    
    # Meilleur jour
    meilleur_jour_idx = df.groupby("day_of_week")["traffic_volume"].mean().idxmin()
    pire_jour_idx = df.groupby("day_of_week")["traffic_volume"].mean().idxmax()
    meilleur_jour = jours[meilleur_jour_idx]
    pire_jour = jours[pire_jour_idx]
    
    # Trafic par heure spécifique
    heure_detectee = None
    for h in range(24):
        if str(h)+"h" in q or f"{h} h" in q or f"heure {h}" in q or f"{h}:00" in q:
            heure_detectee = h
            break
    
    # Jour détecté
    jour_detecte = None
    for i, j in enumerate(jours):
        if j.lower() in q:
            jour_detecte = i
            break
    
    # Mots clés météo
    meteo_detectee = None
    for m in meteos:
        if m.lower() in q:
            meteo_detectee = m
            break

    # === Réponses ===
    
    if any(x in q for x in ["meilleur moment", "meilleur heure", "quand partir", "quand voyager", "heure ideale", "heure idéale", "éviter", "eviter"]):
        trafic_h = df.groupby("hour")["traffic_volume"].mean()
        top3 = trafic_h.nsmallest(3).index.tolist()
        return (f"Voici mes recommandations pour voyager :\n\n"
                f"Les 3 meilleures heures : {top3[0]}h, {top3[1]}h, {top3[2]}h\n"
                f"À éviter absolument : {pire_heure}h (pic maximum)\n\n"
                f"Conseil : Partez tôt le matin (avant 6h) ou tard le soir (après 20h) !")

    elif any(x in q for x in ["pire", "plus fort", "maximum", "embouteillage", "bouchon"]):
        return (f"Les heures avec le plus de trafic :\n\n"
                f"Pire heure : {pire_heure}h\n"
                f"Pire jour : {pire_jour}\n"
                f"Trafic maximum enregistré : {trafic_max:,} véhicules/heure\n\n"
                f"Évitez {pire_jour} à {pire_heure}h !")

    elif any(x in q for x in ["calme", "moins de trafic", "tranquille", "vide", "faible"]):
        return (f"Les moments les plus calmes :\n\n"
                f"Meilleure heure : {meilleure_heure}h\n"
                f"Meilleur jour : {meilleur_jour}\n"
                f"Trafic minimum : {trafic_min:,} véhicules/heure\n\n"
                f"Le {meilleur_jour} à {meilleure_heure}h est le moment idéal !")

    elif any(x in q for x in ["week-end", "weekend", "samedi", "dimanche"]):
        trafic_we = int(df[df["is_weekend"]==1]["traffic_volume"].mean())
        trafic_sem = int(df[df["is_weekend"]==0]["traffic_volume"].mean())
        diff = int(((trafic_sem - trafic_we) / trafic_sem) * 100)
        return (f"Comparaison Week-end vs Semaine :\n\n"
                f"Trafic moyen semaine : {trafic_sem:,} véhicules/heure\n"
                f"Trafic moyen week-end : {trafic_we:,} véhicules/heure\n\n"
                f"Le week-end est {diff}% moins chargé que la semaine !")

    elif any(x in q for x in ["meteo", "météo", "pluie", "neige", "nuage", "soleil"]) or meteo_detectee:
        if meteo_detectee:
            trafic_meteo = int(df[df["weather_main"]==meteo_detectee]["traffic_volume"].mean())
            return f"Trafic moyen par temps {meteo_detectee} : {trafic_meteo:,} véhicules/heure"
        else:
            top_meteo = df.groupby("weather_main")["traffic_volume"].mean().sort_values(ascending=False)
            return (f"Impact de la météo sur le trafic :\n\n"
                    + "\n".join([f"{m} : {int(v):,} véh/h" for m, v in top_meteo.items()]))

    elif heure_detectee is not None:
        trafic_h = int(df[df["hour"]==heure_detectee]["traffic_volume"].mean())
        niveau = "élevé" if trafic_h > 4000 else "modéré" if trafic_h > 2000 else "faible"
        conseil = "Évitez cette heure si possible !" if trafic_h > 4000 else "Bonne heure pour voyager !" if trafic_h < 2000 else "Trafic acceptable."
        if jour_detecte is not None:
            trafic_jh = int(df[(df["hour"]==heure_detectee) & (df["day_of_week"]==jour_detecte)]["traffic_volume"].mean())
            return (f"Trafic {jours[jour_detecte]} à {heure_detectee}h :\n\n"
                    f"Volume moyen : {trafic_jh:,} véhicules/heure\n"
                    f"Niveau : {niveau}\n"
                    f"Conseil : {conseil}")
        return (f"Trafic à {heure_detectee}h :\n\n"
                f"Volume moyen : {trafic_h:,} véhicules/heure\n"
                f"Niveau : {niveau}\n"
                f"Conseil : {conseil}")

    elif jour_detecte is not None:
        trafic_j = int(df[df["day_of_week"]==jour_detecte]["traffic_volume"].mean())
        return (f"Trafic le {jours[jour_detecte]} :\n\n"
                f"Volume moyen : {trafic_j:,} véhicules/heure\n"
                f"Meilleure heure ce jour : {df[df['day_of_week']==jour_detecte].groupby('hour')['traffic_volume'].mean().idxmin()}h")

    elif any(x in q for x in ["moyenne", "moyen", "général", "general", "statistique"]):
        return (f"Statistiques générales du trafic :\n\n"
                f"Trafic moyen : {trafic_moyen:,} véhicules/heure\n"
                f"Maximum : {trafic_max:,} véhicules/heure\n"
                f"Minimum : {trafic_min:,} véhicules/heure\n"
                f"Meilleur jour : {meilleur_jour}\n"
                f"Pire jour : {pire_jour}")

    elif any(x in q for x in ["bonjour", "salut", "hello", "bonsoir"]):
        return ("Bonjour ! Je suis votre assistant trafic routier.\n\n"
                "Je peux vous aider avec :\n"
                "- Le meilleur moment pour voyager\n"
                "- Le trafic à une heure précise\n"
                "- L impact de la météo\n"
                "- Les statistiques générales\n\n"
                "Posez-moi votre question !")

    else:
        return (f"Je n ai pas compris votre question. Essayez par exemple :\n\n"
                f"- Quel est le meilleur moment pour voyager ?\n"
                f"- Quel trafic à 8h ?\n"
                f"- Quel jour est le plus calme ?\n"
                f"- Impact de la pluie sur le trafic ?\n"
                f"- Statistiques générales")

app = dash.Dash(__name__)
app.title = "Dashboard Trafic Routier"

app.layout = html.Div(style={"fontFamily":"Arial","padding":"20px","backgroundColor":"#f8f9fa"}, children=[
    html.H1("Dashboard Trafic Routier", style={"textAlign":"center","color":"#2c3e50"}),
    html.Hr(),

    html.H2("Analyse du trafic"),
    html.Div([
        html.Label("Filtrer par meteo :"),
        dcc.Dropdown(id="meteo-filter",
            options=[{"label":m,"value":m} for m in meteos],
            value=None, placeholder="Toutes les conditions", clearable=True)
    ], style={"width":"40%","marginBottom":"20px"}),
    html.Div([
        dcc.Graph(id="graph-heure", style={"width":"50%","display":"inline-block"}),
        dcc.Graph(id="graph-jour", style={"width":"50%","display":"inline-block"}),
    ]),
    dcc.Graph(id="graph-heatmap"),
    html.Hr(),

    html.H2("Prediction en temps reel"),
    html.Div([
        html.Label("Heure :"),
        dcc.Slider(0,23,1,value=8,marks={i:str(i) for i in range(0,24,2)},id="input-hour"),
        html.Br(),
        html.Label("Jour :"),
        dcc.Dropdown(id="input-day",
            options=[{"label":jours[i],"value":i} for i in range(7)],
            value=0, style={"width":"30%"}),
        html.Br(),
        html.Label("Mois :"),
        dcc.Slider(1,12,1,value=6,marks={i:str(i) for i in range(1,13)},id="input-month"),
        html.Br(),
        html.Label("Temperature (C) :"),
        dcc.Slider(-20,40,1,value=15,marks={i:str(i) for i in range(-20,41,10)},id="input-temp"),
        html.Br(),
        html.Label("Meteo :"),
        dcc.Dropdown(id="input-meteo",
            options=[{"label":m,"value":m} for m in meteos],
            value="Clear", style={"width":"30%"}),
        html.Br(),
        html.Label("Jour ferie ?"),
        dcc.RadioItems(id="input-holiday",
            options=[{"label":"Non","value":0},{"label":"Oui","value":1}],
            value=0, inline=True),
        html.Br(),
        html.Div(id="prediction-output",
            style={"fontSize":"24px","fontWeight":"bold","color":"#27ae60",
                   "textAlign":"center","padding":"20px","backgroundColor":"white",
                   "borderRadius":"10px","border":"2px solid #27ae60"})
    ], style={"backgroundColor":"white","padding":"20px","borderRadius":"10px"}),
    html.Hr(),

    html.H2("Assistant IA — Posez votre question", style={"color":"#8e44ad"}),
    html.Div([
        html.Div([
            html.Div(id="chat-history",
                style={"height":"350px","overflowY":"auto","padding":"15px",
                       "backgroundColor":"#f8f9fa","borderRadius":"10px",
                       "border":"1px solid #ddd","marginBottom":"15px"}),
            html.Div([
                dcc.Input(id="chat-input", type="text",
                    placeholder="Ex: Quel est le meilleur moment pour voyager ?",
                    style={"width":"80%","padding":"10px","borderRadius":"8px",
                           "border":"1px solid #ddd","fontSize":"14px"}),
                html.Button("Envoyer", id="chat-btn",
                    style={"width":"18%","marginLeft":"2%","padding":"10px",
                           "backgroundColor":"#8e44ad","color":"white",
                           "border":"none","borderRadius":"8px",
                           "cursor":"pointer","fontSize":"14px"})
            ], style={"display":"flex"})
        ])
    ], style={"backgroundColor":"white","padding":"20px","borderRadius":"10px"}),
])

@app.callback(
    Output("graph-heure","figure"),
    Output("graph-jour","figure"),
    Output("graph-heatmap","figure"),
    Input("meteo-filter","value"))
def update_graphs(meteo):
    dff = df[df["weather_main"]==meteo] if meteo else df
    fig1 = px.bar(dff.groupby("hour")["traffic_volume"].mean().reset_index(),
        x="hour",y="traffic_volume",title="Trafic moyen par heure",
        color_discrete_sequence=["#3498db"])
    fig2 = px.bar(dff.groupby("day_of_week")["traffic_volume"].mean().reset_index(),
        x="day_of_week",y="traffic_volume",title="Trafic moyen par jour",
        color_discrete_sequence=["#e74c3c"])
    fig2.update_xaxes(tickvals=list(range(7)),ticktext=jours)
    pivot = dff.pivot_table(values="traffic_volume",index="day_of_week",columns="hour",aggfunc="mean")
    fig3 = px.imshow(pivot,title="Carte thermique",color_continuous_scale="YlOrRd",y=jours)
    return fig1,fig2,fig3

@app.callback(
    Output("prediction-output","children"),
    Input("input-hour","value"),
    Input("input-day","value"),
    Input("input-month","value"),
    Input("input-temp","value"),
    Input("input-meteo","value"),
    Input("input-holiday","value"))
def predict(hour,day,month,temp,meteo,holiday):
    is_weekend = 1 if day >= 5 else 0
    weather_enc = le.transform([meteo])[0] if meteo in le.classes_ else 0
    X_input = pd.DataFrame([[hour,day,month,temp,0,0,50,holiday,is_weekend,weather_enc]],
        columns=["hour","day_of_week","month","temp_celsius",
                 "rain_1h","snow_1h","clouds_all","is_holiday","is_weekend","weather_encoded"])
    pred = model.predict(X_input)[0]
    return f"Trafic predit : {int(pred):,} vehicules/heure"

@app.callback(
    Output("chat-history","children"),
    Input("chat-btn","n_clicks"),
    State("chat-input","value"),
    State("chat-history","children"),
    prevent_initial_call=True)
def update_chat(n_clicks, message, historique):
    if not message:
        return historique or []
    
    historique = historique or []
    reponse = analyser_question(message)
    
    msg_user = html.Div([
        html.Span("Vous : ", style={"fontWeight":"bold","color":"#2c3e50"}),
        html.Span(message)
    ], style={"backgroundColor":"#e8f4f8","padding":"10px","borderRadius":"8px",
              "marginBottom":"8px","borderLeft":"4px solid #3498db"})
    
    msg_agent = html.Div([
        html.Span("Agent IA : ", style={"fontWeight":"bold","color":"#8e44ad"}),
        html.Span(reponse, style={"whiteSpace":"pre-line"})
    ], style={"backgroundColor":"#f5eef8","padding":"10px","borderRadius":"8px",
              "marginBottom":"8px","borderLeft":"4px solid #8e44ad"})
    
    return list(historique) + [msg_user, msg_agent]

if __name__ == "__main__":
    app.run(debug=False, port=8050)
'''

with open('dashboard.py', 'w', encoding='utf-8') as f:
    f.write(code)
print("✅ Dashboard avec Agent IA créé !")

✅ Dashboard avec Agent IA créé !


In [15]:
import subprocess
import time

python_anaconda = 'C:\\Users\\user\\anaconda3\\python.exe'

process = subprocess.Popen(
    [python_anaconda, 'dashboard.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(10)
print("✅ Dashboard lancé !")
print("👉 Ouvre Chrome sur : http://127.0.0.1:8050")

✅ Dashboard lancé !
👉 Ouvre Chrome sur : http://127.0.0.1:8050


In [16]:
import os
import subprocess
import time

# Tuer uniquement le processus dashboard (pas Jupyter)
os.system('taskkill /f /fi "WINDOWTITLE eq dashboard*" /im python.exe')
time.sleep(2)

# Relancer avec le bon Python
python_anaconda = 'C:\\Users\\user\\anaconda3\\python.exe'
process = subprocess.Popen([python_anaconda, 'dashboard.py'])
time.sleep(10)
print("✅ Nouveau dashboard lancé !")
print("👉 Va sur : http://127.0.0.1:8050")

✅ Nouveau dashboard lancé !
👉 Va sur : http://127.0.0.1:8050


In [1]:
!git config --global user.email "oumaimaelatiki@gmail.com"
!git config --global user.name "oumaimaelatiki"

In [2]:
!git config --global user.email "elatikioumaima@gmail.com"
!git config --global user.name "oumaimaelatiki"

In [3]:
!git init
!git remote add origin https://github.com/oumaimaelatiki/pfa-traffic-prediction.git

Initialized empty Git repository in C:/Users/user/.git/


In [4]:
!git add .
!git commit -m "Version finale PFA Trafic Casa"
!git push -u origin main --force

error: open("AppData/Local/Comms/UnistoreDB/USS.jtx"): Permission denied
error: unable to index file 'AppData/Local/Comms/UnistoreDB/USS.jtx'
fatal: adding files failed


On branch master

Initial commit

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.anaconda/
	.bash_history
	.cache/
	.cagent/
	.claude.json
	.claude/
	.conda/
	.condarc
	.config/
	.continuum/
	.dacfx/
	.docker/
	.dotnet/
	.eclipse/
	.expo/
	.gitconfig
	.ipynb_checkpoints/
	.ipython/
	.jupyter/
	.matplotlib/
	.ollama/
	.oracle_jre_usage/
	.p2/
	.packettracer
	.spyder-py3/
	.streamlit/
	.trae-aicc/
	.trae/
	.vscode-shared/
	.vscode/
	AppData/
	Cisco Packet Tracer 8.2.2/
	Contacts/
	Dataset for traffic analysis in Casablanca, Morocco.xlsx
	Desktop/
	Documents/
	Downloads/
	Favorites/
	Jedi/
	Links/
	Metro_Interstate_Traffic_Volume.csv
	Microsoft/
	MonApp/
	MonAppNav/
	Music/
	NTUSER.DAT
	NTUSER.DAT{e0cabf3e-10a1-11f0-bb75-c9859b22b0af}.TM.blf
	NTUSER.DAT{e0cabf3e-10a1-11f0-bb75-c9859b22b0af}.TMContainer00000000000000000001.regtrans-ms
	NTUSER.DAT{e0cabf3e-10a1-11f0-bb75-c9859b22b0af}.TMContainer00000000000000000002.regtrans-ms
	OneDrive/
	Projet_Trafic_

error: src refspec main does not match any
error: failed to push some refs to 'https://github.com/oumaimaelatiki/pfa-traffic-prediction.git'
